# PSSN environment validation

This notebook prepares a fresh Google Colab GPU runtime, loads the unquantized `Qwen/Qwen3-4B` model using BF16 when supported (FP16 otherwise), and verifies inference.

In [ ]:
from pathlib import Path
import os

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    !git clone {REPOSITORY_URL} {REPOSITORY_DIR}
os.chdir(REPOSITORY_DIR)
print(f"Working directory: {Path.cwd()}")

In [ ]:
%pip install -q -r requirements.txt

If the next cell reports that CUDA is unavailable, choose **Runtime → Change runtime type → GPU** and restart from the first cell.

In [ ]:
import json
import logging

logging.basicConfig(level=logging.INFO)
from src.utils.environment import collect_environment_info

environment = collect_environment_info()
print(json.dumps(environment, indent=2))
assert environment["cuda_available"], "Enable a Colab GPU runtime before continuing."

In [ ]:
from src.utils.environment import load_qwen_model

tokenizer, model, precision = load_qwen_model()
print(f"Loaded Qwen/Qwen3-4B with {precision} precision on {model.device}")

In [ ]:
import torch

prompt = "What is 17 + 25? Give a short explanation and the answer."
messages = [{"role": "user", "content": prompt}]
rendered_prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=True
)
encoded = tokenizer(
    [rendered_prompt], return_tensors="pt", add_special_tokens=False
)
input_ids = torch.as_tensor(
    encoded["input_ids"], dtype=torch.long, device=model.device
)
model_inputs = {"input_ids": input_ids}
if encoded.get("attention_mask") is not None:
    model_inputs["attention_mask"] = torch.as_tensor(
        encoded["attention_mask"], dtype=torch.long, device=model.device
    )
input_token_count = input_ids.size(-1)
with torch.inference_mode():
    output_ids = model.generate(
        **model_inputs, max_new_tokens=1024, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
response = tokenizer.decode(
    output_ids[0, input_token_count:], skip_special_tokens=True
).strip()
print(response)
assert response.strip(), "Expected the model to generate a non-empty response."

## Persist the research environment

The next cell saves the complete environment snapshot, installed-package versions, model/tokenizer revision metadata, and validation response. Set `PERSIST_TO_GOOGLE_DRIVE=False` to store only in the temporary Colab runtime. Do not store access tokens or other secrets in these artifacts.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

PERSIST_TO_GOOGLE_DRIVE = True
if PERSIST_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/PSSN2/artifacts")
else:
    ARTIFACT_ROOT = REPOSITORY_DIR / "artifacts"

snapshot_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
snapshot_dir = ARTIFACT_ROOT / "environment" / snapshot_id
snapshot_dir.mkdir(parents=True, exist_ok=False)

packages = {}
for package in ["torch", "transformers", "datasets", "accelerate", "safetensors", "sentencepiece"]:
    try:
        packages[package] = version(package)
    except PackageNotFoundError:
        packages[package] = None

def revision_of(component):
    config = getattr(component, "config", None)
    init_kwargs = getattr(component, "init_kwargs", {})
    return getattr(config, "_commit_hash", None) or init_kwargs.get("_commit_hash")

git_commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True
).stdout.strip()
snapshot = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "repository_url": REPOSITORY_URL,
    "repository_commit": git_commit,
    "environment": environment,
    "packages": packages,
    "model": {
        "model_id": "Qwen/Qwen3-4B",
        "model_revision": revision_of(model),
        "tokenizer_revision": revision_of(tokenizer),
        "precision": precision,
        "device": str(model.device),
        "quantized": False,
    },
    "validation": {
        "prompt": "What is 17 + 25? Give a short explanation and the answer.",
        "max_new_tokens": 1024,
        "response": response,
    },
}
(snapshot_dir / "environment_snapshot.json").write_text(
    json.dumps(snapshot, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
pip_freeze = subprocess.run(
    [sys.executable, "-m", "pip", "freeze"],
    capture_output=True, text=True, check=True,
).stdout
(snapshot_dir / "pip_freeze.txt").write_text(pip_freeze, encoding="utf-8")
print(f"Environment artifacts saved to: {snapshot_dir}")